# Phase 7 — Full Benchmark & Evaluation
**RealCentric Generator-Agnostic Deepfake Detection**  
SVKM AI/ML HPC Cluster

Full evaluation: 3 datasets × 3 detection modes + robustness sweep + ablation study.

**Input:** All checkpoints + feature matrices  
**Output:** `/data/mpstme-naman/deepfake_detection/results/benchmark_summary.csv`, `robustness.csv`, `ablation.csv`  
**Estimated time:** 30–60 minutes

## Step 1 — Load All Trained Components

In [ ]:
import sys, numpy as np, pickle
sys.path.insert(0, '/data/mpstme-naman/deepfake_detection')
from pathlib import Path

BASE     = Path('/data/mpstme-naman/deepfake_detection')
FEAT_DIR = BASE / 'data' / 'features'
CKPT_DIR = BASE / 'checkpoints'
RES_DIR  = BASE / 'results'; RES_DIR.mkdir(parents=True, exist_ok=True)

from config.config_loader          import load_config
from src.features.extractor        import FeatureFusionPipeline
from src.models.mlp_classifier     import MLPTrainer
from src.models.one_class_ensemble import OneClassEnsemble
from src.models.autoencoder        import DeepfakeExplainer

cfg = load_config()

pipeline = FeatureFusionPipeline(cfg=cfg, backbone='resnet18')
with open(CKPT_DIR/'pipeline_state.pkl','rb') as f: pipeline.set_state(pickle.load(f))
print('  ✓  Pipeline')

mlp = MLPTrainer(cfg=cfg, input_dim=pipeline.output_dim)
mlp.load_checkpoint(str(CKPT_DIR/'mlp_supervised_best.pt'))
print('  ✓  MLP')

ensemble = OneClassEnsemble(cfg=cfg)
ensemble.load(str(CKPT_DIR/'ensemble.pkl'))
print(f'  ✓  Ensemble  τ={ensemble.threshold:.4f}')

explainer = DeepfakeExplainer(cfg=cfg)
try:
    explainer.load_autoencoder(str(CKPT_DIR/'autoencoder_best.pt'))
    print('  ✓  Autoencoder')
except Exception as e:
    print(f'  ○  Autoencoder not found ({e}) — ELA-only mode')

print('\n  All components loaded!')

## Step 2 — Load Feature Matrices

In [ ]:
datasets = {}
for name, zf, yf in [('CelebDF (test)','Z_test.npy','y_test.npy'),
                     ('FaceForensics++','Z_ff.npy','y_ff.npy'),
                     ('Stable Diffusion','Z_sd.npy','y_sd.npy')]:
    Z = np.load(FEAT_DIR/zf); y = np.load(FEAT_DIR/yf)
    datasets[name] = {'Z':Z,'y':y}
    print(f'  {name:<22} Z={Z.shape}  real={(y==0).sum():,}  fake={(y==1).sum():,}')

## Step 3 — Evaluate All Modes × All Datasets

In [ ]:
from src.evaluation.benchmark import _binary_metrics, _roc_auc

all_results = {}
for ds_name, data in datasets.items():
    Z, y   = data['Z'], data['y']
    both   = len(np.unique(y)) == 2
    res    = {}

    probs = mlp.predict_proba(Z); preds = (probs>=0.5).astype(int)
    res['Supervised MLP']    = _binary_metrics(y,preds,probs) if both else {'detection_rate':float((preds==1).mean()*100)}

    scores = ensemble.score_features(Z); ep = (scores>ensemble.threshold).astype(int)
    res['One-Class Ensemble'] = _binary_metrics(y,ep,scores) if both else {'detection_rate':float((ep==1).mean()*100)}

    mah = Z[:,3]; mp = (mah>np.percentile(mah,50)).astype(int)
    if both: res['Mahalanobis Baseline'] = _binary_metrics(y,mp,mah)

    all_results[ds_name] = res

print('='*74)
print(f'  {"Dataset":<22} {"Mode":<25} {"ACC":>7}  {"AUC":>7}  {"F1":>7}')
print('='*74)
for ds, modes in all_results.items():
    for mode, m in modes.items():
        if 'accuracy' in m:
            print(f'  {ds:<22} {mode:<25} {m["accuracy"]:>6.2f}%  {m.get("auc",0):>7.4f}  {m.get("f1",0):>7.4f}')
        else:
            print(f'  {ds:<22} {mode:<25} detection={m["detection_rate"]:>5.1f}%  (fake-only)')
    print('-'*74)

## Step 4 — Robustness Test (JPEG / Blur / Resize)

In [ ]:
import cv2
from src.evaluation.benchmark import _roc_auc
PROC = BASE / 'data' / 'processed'

def load_n(folder, n=200):
    return [x for x in [cv2.imread(str(p)) for p in sorted(Path(folder).glob('*.png'))[:n]] if x is not None]

def degrade(img, t, p):
    if t=='jpeg':
        _,b=cv2.imencode('.jpg',img,[cv2.IMWRITE_JPEG_QUALITY,int(p)]); return cv2.imdecode(b,cv2.IMREAD_COLOR)
    if t=='blur':
        k=int(p)*2+1; return cv2.GaussianBlur(img,(k,k),0)
    if t=='resize':
        H,W=img.shape[:2]; s=cv2.resize(img,(max(1,int(W*p)),max(1,int(H*p)))); return cv2.resize(s,(W,H))
    return img

ri=load_n(PROC/'celebdf'/'real',200); fi=load_n(PROC/'celebdf'/'fake',200)
dl=np.array([0]*len(ri)+[1]*len(fi))
GRID={'jpeg':[90,70,50,30],'blur':[1,3,5],'resize':[0.75,0.5,0.25]}
rob={}

print(f'  {"Degradation":<20} {"MLP AUC":>9}  {"Ensemble AUC":>14}')
print('  '+'-'*46)
for dt,levels in GRID.items():
    rob[dt]={}
    for lvl in levels:
        imgs=[degrade(img,dt,lvl) for img in ri+fi]
        Z=pipeline.extract_batch(imgs,normalise=True,cnn_batch_size=64)
        ma=_roc_auc(dl,mlp.predict_proba(Z))
        oa=_roc_auc(dl,ensemble.score_features(Z))
        rob[dt][str(lvl)]={'mlp_auc':round(ma,4),'occ_auc':round(oa,4)}
        print(f'  {dt}_{lvl:<16} {ma:>9.4f}  {oa:>14.4f}')

## Step 5 — Ablation Study

In [ ]:
from src.evaluation.benchmark import AblationStudy
Z_test = np.load(FEAT_DIR/'Z_test.npy'); y_test = np.load(FEAT_DIR/'y_test.npy')
print('Ablation — contribution of each feature group to AUC:')
print('='*55)
abl = AblationStudy(mlp_trainer=mlp); res = abl.run(Z_test, y_test)
print()
print(f'  {"Feature Group":<18} {"Full AUC":>9}  {"Ablated":>9}  {"Drop":>8}')
print('  '+'-'*50)
for g,r in res.items():
    print(f'  {g:<18} {r["full_auc"]:>9.4f}  {r["ablated_auc"]:>9.4f}  {r["auc_drop"]:>+8.4f}')

## Step 6 — Save All Result Files

In [ ]:
import csv

def safe(v):
    try: return float(v)
    except: return str(v) if v else ''

# benchmark_summary.csv
p = RES_DIR/'benchmark_summary.csv'
with open(p,'w',newline='') as f:
    w=csv.writer(f); w.writerow(['Dataset','Mode','Accuracy','AUC','F1','Precision','Recall'])
    for ds,modes in all_results.items():
        for mode,m in modes.items():
            w.writerow([ds,mode,safe(m.get('accuracy')),safe(m.get('auc')),
                        safe(m.get('f1')),safe(m.get('precision')),safe(m.get('recall'))])
print(f'  ✓  {p}')

# robustness.csv
p = RES_DIR/'robustness.csv'
with open(p,'w',newline='') as f:
    w=csv.writer(f); w.writerow(['Degradation','Level','MLP_AUC','Ensemble_AUC'])
    for dt,lvls in rob.items():
        for lvl,r in lvls.items(): w.writerow([dt,lvl,r['mlp_auc'],r['occ_auc']])
print(f'  ✓  {p}')

# ablation.csv
p = RES_DIR/'ablation.csv'
with open(p,'w',newline='') as f:
    w=csv.writer(f); w.writerow(['Feature_Group','Full_AUC','Ablated_AUC','AUC_Drop'])
    for g,r in res.items(): w.writerow([g,r['full_auc'],r['ablated_auc'],r['auc_drop']])
print(f'  ✓  {p}')

## Step 7 — Robustness Plot

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Robustness: AUC vs Degradation Level', fontsize=13)
for ax, dt in zip(axes, ['jpeg','blur','resize']):
    lvls = sorted(rob[dt].keys(), key=float)
    ax.plot(lvls, [rob[dt][l]['mlp_auc'] for l in lvls], 'o-', color='steelblue', label='MLP Supervised')
    ax.plot(lvls, [rob[dt][l]['occ_auc'] for l in lvls], 's-', color='crimson',   label='One-Class Ensemble')
    ax.set_title(dt.upper()); ax.set_ylim(0.4,1.0); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
out = RES_DIR/'robustness_plot.png'
plt.savefig(out, dpi=120, bbox_inches='tight'); plt.show()
print(f'  ✓  {out}')

## ✅ Phase 7 Complete

```
/data/mpstme-naman/deepfake_detection/results/
├── benchmark_summary.csv
├── robustness.csv
├── ablation.csv
└── robustness_plot.png
```

**Next:** Open `07_inference.ipynb`